# Data Exploration — NASA POWER (Peninsular Malaysia 2020)

This notebook connects to the public NASA POWER S3 bucket, extracts SYN1DEG
and MERRA2 variables for Peninsular Malaysia (year 2020), combines them into
a single DataFrame, and performs basic data exploration.

## 1. Library Imports

In [ ]:
import pandas as pd
import s3fs
import xarray as xr

## 2. Connect to S3 and Open Zarr Datasets

In [ ]:
fs = s3fs.S3FileSystem(anon=True)

SYN1DEG_DAILY = "s3://nasa-power/syn1deg/temporal/power_syn1deg_daily_temporal_utc.zarr"
MERRA2_DAILY  = "s3://nasa-power/merra2/temporal/power_merra2_daily_temporal_utc.zarr"

syn_ds   = xr.open_zarr(SYN1DEG_DAILY, storage_options={"anon": True})
merra_ds = xr.open_zarr(MERRA2_DAILY,  storage_options={"anon": True})

print("SYN1DEG dataset:\n", syn_ds)
print("\nMERRA2 dataset:\n", merra_ds)

## 3. Region and Time Filtering — Peninsular Malaysia (2020)

In [ ]:
bounds = {
    "lat_min": 1,
    "lat_max": 8,
    "lon_min": 98,
    "lon_max": 105
}

syn_region = syn_ds.sel(
    lat=slice(bounds["lat_min"], bounds["lat_max"]),
    lon=slice(bounds["lon_min"], bounds["lon_max"]),
    time=slice("2020-01-01", "2020-12-31")
)

merra_region = merra_ds.sel(
    lat=slice(bounds["lat_min"], bounds["lat_max"]),
    lon=slice(bounds["lon_min"], bounds["lon_max"]),
    time=slice("2020-01-01", "2020-12-31")
)

print("SYN1DEG region time steps:", syn_region.dims["time"])
print("MERRA2  region time steps:", merra_region.dims["time"])

## 4. Variable Selection and Spatial Aggregation

In [ ]:
SYN1DEG_VARS = ["ALLSKY_SFC_SW_DWN", "CLOUD_AMT", "AOD_55", "PW", "ALLSKY_KT", "PSH"]
MERRA2_VARS  = ["RH2M", "WS2M", "T2M_MAX", "PS"]

# Spatial mean (lat, lon) → daily time series
syn_data = {
    var: syn_region[var].mean(dim=["lat", "lon"]).to_series()
    for var in SYN1DEG_VARS
}
merra_data = {
    var: merra_region[var].mean(dim=["lat", "lon"]).to_series()
    for var in MERRA2_VARS
}

df_syn   = pd.DataFrame(syn_data)
df_merra = pd.DataFrame(merra_data)

df_combined = pd.concat([df_syn, df_merra], axis=1).dropna()
df_combined.index.name = "time"
print("Combined shape:", df_combined.shape)
df_combined.head()

## 5. Basic Information

In [ ]:
df_combined.info()

## 6. Summary Statistics

In [ ]:
df_combined.describe().T.round(3)

## 7. Correlation Matrix

In [ ]:
df_combined.corr().round(3)

## 8. Missing Values

In [ ]:
missing = df_combined.isnull().sum().rename("missing_count")
missing_pct = (df_combined.isnull().mean() * 100).rename("missing_%")
pd.concat([missing, missing_pct], axis=1)